# P9 — Pipeline Fruits sur EMR Studio (kernel PySpark)

**Conformité RGPD** : ce notebook s'exécute sur un cluster EMR situé dans la région AWS `eu-west-3` (Paris). Les données stockées sur S3 sont également dans cette région. Aucune donnée ne quitte le territoire européen pendant le traitement.

**Architecture Big Data utilisée** :
- **Stockage** : Amazon S3 (objet, scalable, chiffré au repos via SSE-S3)
- **Calcul distribué** : Apache Spark sur Amazon EMR (managed Hadoop/YARN)
- **Orchestration** : EMR Studio + Livy pour l'interface notebook → cluster
- **Featurization** : MobileNetV2 (TensorFlow) distribué via `pandas_udf`
- **Réduction de dimension** : Spark ML PCA

## 0. Configuration de la session Spark

⚠️ **Cellule à exécuter EN PREMIER** — elle redémarre le kernel avec ces configs.

Ajuster `spark.executor.instances` au nombre de workers de ton cluster.

In [ ]:
%%configure -f
{
    "conf": {
        "spark.executor.memory": "20g",
        "spark.executor.cores": "4",
        "spark.executor.instances": "6",
        "spark.driver.memory": "8g",
        "spark.sql.execution.arrow.pyspark.enabled": "true",
        "spark.sql.execution.arrow.maxRecordsPerBatch": "1024",
        "spark.sql.parquet.writeLegacyFormat": "true",
        "spark.sql.shuffle.partitions": "64"
    }
}

## 1. Imports

Le kernel PySpark expose déjà `spark` et `sc`. Pas besoin de `SparkSession.builder`.

Les imports ci-dessous sont exécutés **côté driver** sur le master EMR.

In [ ]:
import io
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model

from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split, udf
from pyspark.ml.feature import PCA, VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT

# Vérification de la session déjà fournie par le kernel
print('Spark version :', spark.version)
print('Master :', sc.master)
print('App name :', sc.appName)

## 2. Chemins S3

**RGPD** : ces chemins pointent vers un bucket S3 créé en `eu-west-3` (Paris).

In [ ]:
S3_BUCKET = 's3://nons-bucket-p9-096766144198-eu-west-3-an'

PATH_Data    = f'{S3_BUCKET}/data/fruits-360/Test'
PATH_Result  = f'{S3_BUCKET}/features'
PATH_PCA     = f'{S3_BUCKET}/pca'

print('Source       :', PATH_Data)
print('Features out :', PATH_Result)
print('PCA out      :', PATH_PCA)

## 3. Chargement des images depuis S3

In [ ]:
images = (
    spark.read.format('binaryFile')
    .option('pathGlobFilter', '*.jpg')
    .option('recursiveFileLookup', 'true')
    .load(PATH_Data)
)

# Extraction du label depuis le dossier parent
images = images.withColumn('label', element_at(split(images['path'], '/'), -2))

images.printSchema()
images.select('path', 'label').show(5, truncate=False)

In [ ]:
# Comptage : déclenche un job Spark sur l'ensemble des fichiers
n_total = images.count()
print(f'Nombre total d\'images : {n_total:,}')

### 3 bis. Exploration rapide (optionnelle)

Profite du mode interactif pour comprendre la distribution des classes **avant** de lancer la featurization complète (qui prend des minutes).

In [ ]:
# Top 20 des classes les plus représentées
from pyspark.sql.functions import count
(images.groupBy('label')
       .agg(count('*').alias('n'))
       .orderBy(col('n').desc())
       .show(20, truncate=False))

## 4. MobileNetV2 + broadcast des poids

In [ ]:
# Chargement côté driver
base = MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
new_model = Model(inputs=base.input, outputs=base.layers[-2].output)

# Broadcast des poids vers tous les workers (transfert efficace, 1 fois)
broadcast_weights = sc.broadcast(new_model.get_weights())
print(f'Modèle chargé : {len(new_model.layers)} couches, output dim = {new_model.output_shape[-1]}')

## 5. UDF de featurization distribuée

**Traitement critique pour le passage à l'échelle** : c'est ici que le calcul se distribue. Sans `pandas_udf`, on devrait `collect()` toutes les images sur le driver — impossible au-delà de quelques centaines.

In [ ]:
def model_fn():
    """Reconstruit le modèle sur chaque worker avec les poids broadcastés."""
    m = MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
    for layer in m.layers:
        layer.trainable = False
    nm = Model(inputs=m.input, outputs=m.layers[-2].output)
    nm.set_weights(broadcast_weights.value)
    return nm


def preprocess(content):
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    return preprocess_input(img_to_array(img))


def featurize_series(model, content_series):
    arr = np.stack(content_series.map(preprocess))
    preds = model.predict(arr)
    return pd.Series([p.flatten() for p in preds])


@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    """
    Scalar-Iterator pandas UDF :
    - charge le modèle UNE fois par worker
    - traite tous les batchs envoyés par Spark à ce worker
    """
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

## 6. Featurization → écriture S3

Le `.write` déclenche l'exécution réelle de tout le pipeline (lazy evaluation Spark).

Pendant l'exécution, regarde l'onglet **Spark UI** (lien dans EMR Studio) pour suivre les stages, le shuffle, et l'utilisation des executors.

In [ ]:
features_df = (
    images
    .repartition(64)  # ajuster selon nb_cores * 2-3
    .select(
        col('path'),
        col('label'),
        featurize_udf('content').alias('features')
    )
)

features_df.write.mode('overwrite').parquet(PATH_Result)
print(f'✓ Features écrites : {PATH_Result}')

## 7. PCA — recherche du k optimal

Calcul distribué de la décomposition. On fit à `k=200` pour explorer la courbe, puis on choisit le k qui couvre 90% de la variance.

In [ ]:
df_feat = spark.read.parquet(PATH_Result)

# Conversion array<float> -> Vector pour Spark ML
to_vector = udf(lambda arr: Vectors.dense(arr), VectorUDT())
df_vec = df_feat.withColumn('features_vec', to_vector('features'))

df_vec.printSchema()

In [ ]:
# PCA exploratoire
k_max = 200
pca_explore = PCA(k=k_max, inputCol='features_vec', outputCol='pcaFeatures')
pcaModel = pca_explore.fit(df_vec)

cumValues = pcaModel.explainedVariance.cumsum()

for seuil in [0.80, 0.90, 0.95]:
    mask = cumValues >= seuil
    if mask.any():
        K = int(np.argmax(mask) + 1)
        print(f'  {int(seuil*100)}% de variance → k = {K}')
    else:
        print(f'  {int(seuil*100)}% non atteint (max = {cumValues[-1]*100:.1f}% à k={k_max})')

# Détermination du k retenu (utilisé en cellule suivante)
mask_90 = cumValues >= 0.90
k_optimal = int(np.argmax(mask_90) + 1) if mask_90.any() else k_max
print(f'\n→ k retenu : {k_optimal}')

### Visualisation de la courbe de variance

**Important** : on utilise `%%local` pour faire le plot. Cette magic exécute la cellule **dans ton navigateur** (Python local au notebook) au lieu de l'envoyer au cluster. Pour partager des données entre les deux modes, on utilise `%%send_to_spark` et `%%spark`.

In [ ]:
# Étape 1 : envoyer cumValues du cluster vers le notebook local
# (cumValues est un numpy array côté driver, on le sérialise simplement)
cum_list = cumValues.tolist()
k_opt_val = k_optimal
print('Données prêtes pour le plot local')

In [ ]:
%%local
# Cellule exécutée en local — matplotlib disponible ici
import matplotlib.pyplot as plt
import numpy as np

# Note : pour récupérer cum_list/k_opt_val depuis le cluster, utiliser %%spark -o
# Ici version simplifiée : refaire un calcul local sur un échantillon,
# OU mieux : utiliser sparkmagic %%spark -o cum_df pour exporter en pandas DataFrame.
# Pour la démo, on assume que cum_list a été récupéré.

# Pattern recommandé en pratique :
#   Cellule cluster :  cum_df = pd.DataFrame({'k': range(1, len(cumValues)+1), 'var': cumValues})
#                      → utiliser %%spark -o cum_df pour rapatrier
#   Cellule %%local :  plt.plot(cum_df['k'], cum_df['var'])

print('Pour plot local : voir le pattern dans la cellule markdown ci-dessous')

**Pattern complet pour transférer un DataFrame Spark → pandas local :**

```python
# Cellule 1 (cluster, kernel PySpark par défaut) :
import pandas as pd
cum_df = pd.DataFrame({'k': range(1, len(cumValues)+1), 'var': cumValues})
# Note : si cum_df est petit, on peut le rapatrier directement.
# Pour un gros DataFrame Spark, faire .limit(N).toPandas() avant.
```
```python
%%spark -o cum_df_local -m sample -n 500
# 'cum_df_local' sera dispo dans les cellules %%local
```
```python
%%local
import matplotlib.pyplot as plt
plt.plot(cum_df_local['k'], cum_df_local['var'])
plt.axhline(0.9, color='r', linestyle=':')
plt.show()
```

## 8. PCA finale + écriture S3

In [ ]:
pca_final = PCA(k=k_optimal, inputCol='features_vec', outputCol='pcaFeatures')
model_pca = pca_final.fit(df_vec)

result = model_pca.transform(df_vec).drop('features_vec')
result.show(5, truncate=80)

In [ ]:
# Écriture finale sur S3
result.write.mode('overwrite').parquet(PATH_PCA)
print(f'✓ PCA écrite : {PATH_PCA}')

## 9. Vérification

In [ ]:
df_check = spark.read.parquet(PATH_PCA)
print(f'Lignes : {df_check.count():,}')
df_check.printSchema()
df_check.show(5, truncate=80)

## 10. Note conformité RGPD

**Validations à mentionner dans le rapport :**

| Critère | Validation |
|---|---|
| Stockage en zone européenne | Bucket S3 créé en `eu-west-3` (Paris) |
| Calcul en zone européenne | Cluster EMR provisionné en `eu-west-3` |
| Chiffrement au repos | SSE-S3 activé par défaut sur le bucket |
| Chiffrement en transit | HTTPS/TLS pour S3, communications inter-nœuds dans le VPC |
| Contrôle d'accès | IAM roles spécifiques (`EMR_EC2_DefaultRole`), pas de credentials en clair |
| Pas de fuite hors UE | Aucun service AWS d'autre région appelé dans le pipeline |

Pour un dataset contenant des données personnelles (pas le cas ici avec Fruits-360), il faudrait en plus : journalisation S3 access logs + CloudTrail, anonymisation/pseudonymisation en amont, durée de rétention définie via S3 Lifecycle, et chiffrement KMS avec clé dédiée.